# Bundestagswahl 2021: Wahlkreisdaten Schritt für Schritt aufbereiten

Dieses Notebook zeigt jeden Verarbeitungsschritt offen. Die eigentliche Rechenlogik liegt in kleinen Python-Funktionen, aber keine große Pipeline wird als Blackbox aufgerufen.

Die Wahlbezirksdatei wird zuerst geprüft und anschließend auf Wahlkreise zusammengefasst. Die repräsentative Statistik liefert nur Verteilungen auf Landesebene. Ihre gerundeten absoluten Werte werden deshalb in Anteile umgerechnet und später auf die amtlichen Stimmen aus der Wahlbezirksdatei angewendet.

Die Quellkategorie `m` enthält laut Hinweis der Bundeswahlleitung auch Personen mit dem Geschlechtsmerkmal divers sowie Personen ohne Geschlechtseintrag im Geburtenregister.

## 0. Arbeitsverzeichnis und lokale Dateien

In [1]:
from pathlib import Path
import sys
from dataclasses import asdict

import pandas as pd
from IPython.display import display


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("Das Notebook muss innerhalb des Repository-Ordners laufen.")


ROOT = find_repository_root()
sys.path.insert(0, str(ROOT))
ROOT

PosixPath('/home/nomikron/WebstormProjects/mach-dir-dein-bundestag')

In [2]:
# Diese Namen sind nur lokale Vorschläge. Trage hier die tatsächlich
# heruntergeladenen CSV-Dateien ein.
DISTRICT_RESULTS_CSV = ROOT / "scripts/data/btw21_wbz_ergebnisse.csv"
STATE_DEMOGRAPHICS_CSV = ROOT / "scripts/data/btw21_rws_bst2.csv"
FEDERAL_METHOD_DEMOGRAPHICS_CSV = None  # optionaler lokaler Pfad
OUTPUT_DIRECTORY = ROOT / "scripts/data/generated"

DISTRICT_RESULTS_CSV, STATE_DEMOGRAPHICS_CSV, FEDERAL_METHOD_DEMOGRAPHICS_CSV

(PosixPath('/home/nomikron/WebstormProjects/mach-dir-dein-bundestag/scripts/data/btw21_wbz_ergebnisse.csv'),
 PosixPath('/home/nomikron/WebstormProjects/mach-dir-dein-bundestag/scripts/data/btw21_rws_bst2.csv'),
 None)

In [3]:
from scripts.election_data.btw2021 import read_local_csv
from scripts.election_data.notebook_steps import (
    aggregate_to_constituencies,
    calculate_demographic_profiles,
    calculate_federal_method_weights,
    district_party_columns,
    inspect_district_rows,
    normalize_district_rows,
    normalize_federal_method_rows,
    normalize_state_statistic_rows,
    reshape_federal_method_votes,
    reshape_polling_district_votes,
    reshape_state_statistic_votes,
    select_federal_method_detail_rows,
    select_state_statistic_detail_rows,
    select_usable_district_rows,
)
from scripts.election_data.pipeline import distribute_district_votes, write_vote_entries
from scripts.election_data.profiles import build_state_method_profiles
from scripts.election_data.validation import validate_vote_entries

## 1. Wahlbezirksdatei unverändert einlesen

Zuerst wird nur gelesen. Noch wird keine Zeile entfernt und keine Spalte umbenannt.

In [4]:
raw_districts = read_local_csv(DISTRICT_RESULTS_CSV)
print(f"Zeilen: {len(raw_districts):,}")
print(f"Spalten: {len(raw_districts.columns):,}")
display(raw_districts.head())
display(raw_districts.tail())

Zeilen: 96,300
Spalten: 107


,Wahlkreis,Land,Regierungsbezirk,Kreis,Verbandsgemeinde,Gemeinde,Kennziffer Urnenwahlbezirke nach § 68 BWO,Kennziffer Briefwahlzugehörigkeit,Gemeinde Name,Wahlbezirk,...,Z_Bündnis21,Z_LIEBE,Z_LKR,Z_PdF,Z_LfK,Z_SSW,Z_Team Todenhöfer,Z_UNABHÄNGIGE,Z_Volt,Ungekürzte Wahlbezirksbezeichnung
0,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",01,...,0,0,0,0,0,118,5,0,1,01
1,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",02,...,0,0,0,0,0,130,13,0,3,02
2,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",03,...,0,0,0,0,0,106,4,0,0,03
3,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",04,...,0,0,0,0,0,84,6,0,0,04
4,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",05,...,0,0,0,0,0,50,3,0,1,05


,Wahlkreis,Land,Regierungsbezirk,Kreis,Verbandsgemeinde,Gemeinde,Kennziffer Urnenwahlbezirke nach § 68 BWO,Kennziffer Briefwahlzugehörigkeit,Gemeinde Name,Wahlbezirk,...,Z_Bündnis21,Z_LIEBE,Z_LKR,Z_PdF,Z_LfK,Z_SSW,Z_Team Todenhöfer,Z_UNABHÄNGIGE,Z_Volt,Ungekürzte Wahlbezirksbezeichnung
96295,,,,,,,,,,,...,,,,,,,,,,
96296,,,,,,,,,,,...,,,,,,,,,,
96297,,,,,,,,,,,...,,,,,,,,,,
96298,,,,,,,,,,,...,,,,,,,,,,
96299,,,,,,,,,,,...,,,,,,,,,,


## 2. Wahlkreisnummern prüfen

Bevor `Wahlkreis` in eine Ganzzahl umgewandelt wird, werden alle Zeilen eingeteilt:

- `usable`: enthält eine normale Wahlkreisnummer;
- `missing`: leerer Wert oder eine leere Abschlusszeile;
- `invalid`: nicht leer, aber keine Zahl. Solche Zeilen müssen untersucht werden.

Dadurch ist bei einem Fehler direkt sichtbar, welche Quellzeilen ihn verursachen.

In [5]:
district_row_diagnostics = inspect_district_rows(raw_districts)
display(district_row_diagnostics["status"].value_counts().rename("rows"))

problematic_district_rows = district_row_diagnostics[
    district_row_diagnostics["status"] != "usable"
]
display(problematic_district_rows)

status
usable     94668
missing     1632
Name: rows, dtype: int64

,sourceRow,Wahlkreis,Land,Gemeinde Name,Wahlbezirk,Bezirksart,districtIdCandidate,status
94668,94668,,,,,,<NA>,missing
94669,94669,,,,,,<NA>,missing
94670,94670,,,,,,<NA>,missing
94671,94671,,,,,,<NA>,missing
94672,94672,,,,,,<NA>,missing
...,...,...,...,...,...,...,...,...
96295,96295,,,,,,<NA>,missing
96296,96296,,,,,,<NA>,missing
96297,96297,,,,,,<NA>,missing
96298,96298,,,,,,<NA>,missing


Jetzt werden nur die zuvor sichtbaren leeren Zeilen entfernt. Nicht leere, ungültige Werte führen weiterhin zu einem Fehler mit den konkreten Quellzeilen.

In [6]:
usable_district_rows = select_usable_district_rows(
    raw_districts,
    district_row_diagnostics,
)
print(f"Verwendete Zeilen: {len(usable_district_rows):,}")
display(usable_district_rows.head())

Verwendete Zeilen: 94,668


,Wahlkreis,Land,Regierungsbezirk,Kreis,Verbandsgemeinde,Gemeinde,Kennziffer Urnenwahlbezirke nach § 68 BWO,Kennziffer Briefwahlzugehörigkeit,Gemeinde Name,Wahlbezirk,...,Z_Bündnis21,Z_LIEBE,Z_LKR,Z_PdF,Z_LfK,Z_SSW,Z_Team Todenhöfer,Z_UNABHÄNGIGE,Z_Volt,Ungekürzte Wahlbezirksbezeichnung
0,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",01,...,0,0,0,0,0,118,5,0,1,01
1,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",02,...,0,0,0,0,0,130,13,0,3,02
2,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",03,...,0,0,0,0,0,106,4,0,0,03
3,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",04,...,0,0,0,0,0,84,6,0,0,04
4,001,01,0,01,0000,000,0000,00,"Flensburg, Stadt",05,...,0,0,0,0,0,50,3,0,1,05


## 3. Wahlkreis, Bundesland und Wahlart vereinheitlichen

Die Wahlkreisnummer wird jetzt zu `districtId`. Die Landeskennziffer wird zum ausgeschriebenen Bundesland. Die Bezirksarten werden für die App auf `postal` und `in-person` abgebildet.

In [7]:
normalized_district_rows = normalize_district_rows(usable_district_rows)

display(
    normalized_district_rows[
        ["Wahlkreis", "districtId", "Land", "state", "Bezirksart", "electionMethod"]
    ].head(20)
)
display(normalized_district_rows["electionMethod"].value_counts())

,Wahlkreis,districtId,Land,state,Bezirksart,electionMethod
0,001,1,01,Schleswig-Holstein,0,in-person
1,001,1,01,Schleswig-Holstein,0,in-person
2,001,1,01,Schleswig-Holstein,0,in-person
3,001,1,01,Schleswig-Holstein,0,in-person
4,001,1,01,Schleswig-Holstein,0,in-person
5,001,1,01,Schleswig-Holstein,0,in-person
6,001,1,01,Schleswig-Holstein,0,in-person
7,001,1,01,Schleswig-Holstein,0,in-person
8,001,1,01,Schleswig-Holstein,0,in-person
9,001,1,01,Schleswig-Holstein,0,in-person


electionMethod
in-person    67021
postal       27647
Name: count, dtype: int64

## 4. Parteienfelder der Erst- und Zweitstimmen bestimmen

Die Spalten `E_Gültige`, `E_Ungültige`, `Z_Gültige` und `Z_Ungültige` sind Summen und keine Parteien. Alle übrigen `E_`- beziehungsweise `Z_`-Spalten bleiben erhalten.

In [8]:
first_vote_columns = district_party_columns(normalized_district_rows, "E_")
second_vote_columns = district_party_columns(normalized_district_rows, "Z_")

print(f"Erststimmen-Parteifelder: {len(first_vote_columns)}")
print(f"Zweitstimmen-Parteifelder: {len(second_vote_columns)}")
display(pd.Series(first_vote_columns, name="first vote columns").to_frame())
display(pd.Series(second_vote_columns, name="second vote columns").to_frame())

Erststimmen-Parteifelder: 45
Zweitstimmen-Parteifelder: 40


,first vote columns
0,E_CDU
1,E_SPD
2,E_AfD
3,E_FDP
4,E_DIE LINKE
5,E_GRÜNE
6,E_CSU
7,E_FREIE WÄHLER
8,E_Die PARTEI
9,E_Tierschutzpartei


,second vote columns
0,Z_CDU
1,Z_SPD
2,Z_AfD
3,Z_FDP
4,Z_DIE LINKE
5,Z_GRÜNE
6,Z_CSU
7,Z_FREIE WÄHLER
8,Z_Die PARTEI
9,Z_Tierschutzpartei


## 5. Breite Parteispalten in einzelne Stimmenzeilen umformen

Jede Zeile beschreibt danach genau eine Kombination aus Quellzeile, Wahlkreis, Partei, Stimmenart und Brief-/Urnenwahl. Zu diesem Zeitpunkt liegen die Daten weiterhin auf Wahlbezirksebene vor.

In [9]:
first_polling_district_votes = reshape_polling_district_votes(
    normalized_district_rows,
    prefix="E_",
    vote_type="1",
)
second_polling_district_votes = reshape_polling_district_votes(
    normalized_district_rows,
    prefix="Z_",
    vote_type="2",
)

display(first_polling_district_votes.head(20))
display(second_polling_district_votes.head(20))

,sourceRow,districtId,state,party,voteType,electionMethod,votes
0,0,1,Schleswig-Holstein,AfD,1,in-person,38.0
1,1,1,Schleswig-Holstein,AfD,1,in-person,69.0
2,2,1,Schleswig-Holstein,AfD,1,in-person,44.0
3,3,1,Schleswig-Holstein,AfD,1,in-person,45.0
4,4,1,Schleswig-Holstein,AfD,1,in-person,41.0
5,5,1,Schleswig-Holstein,AfD,1,in-person,38.0
6,6,1,Schleswig-Holstein,AfD,1,in-person,37.0
7,7,1,Schleswig-Holstein,AfD,1,in-person,24.0
8,8,1,Schleswig-Holstein,AfD,1,in-person,24.0
9,9,1,Schleswig-Holstein,AfD,1,in-person,21.0


,sourceRow,districtId,state,party,voteType,electionMethod,votes
0,0,1,Schleswig-Holstein,AfD,2,in-person,35.0
1,1,1,Schleswig-Holstein,AfD,2,in-person,59.0
2,2,1,Schleswig-Holstein,AfD,2,in-person,42.0
3,3,1,Schleswig-Holstein,AfD,2,in-person,47.0
4,4,1,Schleswig-Holstein,AfD,2,in-person,34.0
5,5,1,Schleswig-Holstein,AfD,2,in-person,34.0
6,6,1,Schleswig-Holstein,AfD,2,in-person,38.0
7,7,1,Schleswig-Holstein,AfD,2,in-person,22.0
8,8,1,Schleswig-Holstein,AfD,2,in-person,27.0
9,9,1,Schleswig-Holstein,AfD,2,in-person,27.0


## 6. Wahlbezirke zu Wahlkreisen zusammenfassen

Für die App wird nicht jeder Wahlbezirk gespeichert. Alle Wahlbezirke desselben Wahlkreises werden nach Partei, Stimmenart und Brief-/Urnenwahl addiert.

In [10]:
polling_district_votes = pd.concat(
    [first_polling_district_votes, second_polling_district_votes],
    ignore_index=True,
)
district_totals = aggregate_to_constituencies(polling_district_votes)

print(f"Wahlkreise: {district_totals['districtId'].nunique()}")
print(f"Bundesländer: {district_totals['state'].nunique()}")
print(f"Parteien/Sammelkategorien: {district_totals['party'].nunique()}")
print(f"Wahlkreis-Summen-Zeilen: {len(district_totals):,}")
display(district_totals.head(30))

Wahlkreise: 299
Bundesländer: 16
Parteien/Sammelkategorien: 48
Wahlkreis-Summen-Zeilen: 20,579


,districtId,state,party,voteType,electionMethod,votes
0,1,Schleswig-Holstein,AfD,1,in-person,7936.0
1,1,Schleswig-Holstein,AfD,1,postal,1832.0
2,1,Schleswig-Holstein,AfD,2,in-person,8293.0
3,1,Schleswig-Holstein,AfD,2,postal,2024.0
4,1,Schleswig-Holstein,CDU,1,in-person,29743.0
5,1,Schleswig-Holstein,CDU,1,postal,11978.0
6,1,Schleswig-Holstein,CDU,2,in-person,25562.0
7,1,Schleswig-Holstein,CDU,2,postal,10859.0
8,1,Schleswig-Holstein,DIE LINKE,1,in-person,4701.0
9,1,Schleswig-Holstein,DIE LINKE,1,postal,1843.0


In [11]:
sample_district_id = int(district_totals["districtId"].min())
display(
    district_totals[district_totals["districtId"] == sample_district_id]
    .sort_values(["voteType", "electionMethod", "votes"], ascending=[True, True, False])
    .head(40)
)

,districtId,state,party,voteType,electionMethod,votes
26,1,Schleswig-Holstein,GRÜNE,1,in-person,31751.0
4,1,Schleswig-Holstein,CDU,1,in-person,29743.0
38,1,Schleswig-Holstein,SPD,1,in-person,29588.0
42,1,Schleswig-Holstein,SSW,1,in-person,9280.0
18,1,Schleswig-Holstein,FDP,1,in-person,9034.0
0,1,Schleswig-Holstein,AfD,1,in-person,7936.0
8,1,Schleswig-Holstein,DIE LINKE,1,in-person,4701.0
54,1,Schleswig-Holstein,dieBasis,1,in-person,2526.0
22,1,Schleswig-Holstein,FREIE WÄHLER,1,in-person,1630.0
58,1,Schleswig-Holstein,du.,1,in-person,137.0


## 7. Repräsentative Landesstatistik unverändert einlesen

Die Kommentarzeilen am Anfang beginnen mit `#` und werden beim Einlesen übersprungen. Die eigentliche Tabelle bleibt ansonsten unverändert.

In [12]:
raw_state_statistics = read_local_csv(STATE_DEMOGRAPHICS_CSV, comment="#")
print(f"Zeilen: {len(raw_state_statistics):,}")
print(f"Spalten: {len(raw_state_statistics.columns):,}")
display(raw_state_statistics.head(20))

Zeilen: 798
Spalten: 16


,Land,Erst-/Zweitstimme,Geschlecht,Geburtsjahresgruppe,Summe,Ungültig,CDU,SPD,AfD,FDP,DIE LINKE,GRÜNE,CSU,Sonstige,dar. FREIE WÄHLER,dar. dieBasis
0,Bund,1,Summe,Summe,46854508,492495,10451524,12234690,4695611,4042951,2307536,6469081,2788048,3372572,1316686,734011
1,Bund,1,Summe,1997 – 2003,3524799,25512,434605,667343,218003,592862,276324,816458,132261,361432,106749,48258
2,Bund,1,Summe,1987 – 1996,5992701,44955,835239,1203503,588419,718457,407852,1289831,253494,650951,224446,111570
3,Bund,1,Summe,1977 – 1986,6466965,51452,1197212,1342590,880187,609213,323578,1081171,332210,649352,242994,155238
4,Bund,1,Summe,1962 – 1976,12604097,111747,2721637,3171648,1559505,1051192,523633,1738183,727673,998879,416490,257493
5,Bund,1,Summe,1952 – 1961,8435960,95625,1965006,2607468,882411,546049,394588,944474,553345,446994,205410,105472
6,Bund,1,Summe,1951 und früher,9829986,163203,3297825,3242137,567086,525178,381561,598964,789066,264964,120596,55981
7,Bund,1,m,Summe,22734708,220270,4982359,5737028,2854612,2110369,1107736,2831007,1344645,1546681,613091,294789
8,Bund,1,m,1997 – 2003,1771704,11797,226552,323416,132304,368718,114541,342015,70142,182219,52641,22052
9,Bund,1,m,1987 – 1996,3002958,22548,421268,584832,348089,416227,199337,557623,130911,322122,107029,47848


## 8. Dimensionen der Statistik vereinheitlichen

Bundesland, Erst-/Zweitstimme, Geschlecht und Altersgruppe erhalten die Werte, die später auch im JSON stehen. Summenzeilen bleiben zunächst erhalten und sind an leeren normalisierten Dimensionen erkennbar.

In [13]:
normalized_state_statistics = normalize_state_statistic_rows(raw_state_statistics)

display(
    normalized_state_statistics[
        [
            "Land",
            "state",
            "Erst-/Zweitstimme",
            "voteType",
            "Geschlecht",
            "gender",
            "Geburtsjahresgruppe",
            "ageGroup",
        ]
    ].head(30)
)

,Land,state,Erst-/Zweitstimme,voteType,Geschlecht,gender,Geburtsjahresgruppe,ageGroup
0,Bund,None,1,1,Summe,None,Summe,None
1,Bund,None,1,1,Summe,None,1997 – 2003,18-24
2,Bund,None,1,1,Summe,None,1987 – 1996,25-34
3,Bund,None,1,1,Summe,None,1977 – 1986,35-44
4,Bund,None,1,1,Summe,None,1962 – 1976,45-54
5,Bund,None,1,1,Summe,None,1952 – 1961,55-64
6,Bund,None,1,1,Summe,None,1951 und früher,65+
7,Bund,None,1,1,m,m,Summe,None
8,Bund,None,1,1,m,m,1997 – 2003,18-24
9,Bund,None,1,1,m,m,1987 – 1996,25-34


Die folgenden Zeilen sind Bundes-, Geschlechts-, Alters- oder andere Summen. Sie werden angezeigt, bevor sie für die Detailverteilung entfernt werden.

In [14]:
statistic_dimensions = ["state", "voteType", "gender", "ageGroup"]
summary_statistic_rows = normalized_state_statistics[
    ~normalized_state_statistics[statistic_dimensions].notna().all(axis=1)
]
print(f"Ausgeschlossene Summenzeilen: {len(summary_statistic_rows):,}")
display(
    summary_statistic_rows[
        ["Land", "Erst-/Zweitstimme", "Geschlecht", "Geburtsjahresgruppe"]
    ].head(50)
)

Ausgeschlossene Summenzeilen: 414


,Land,Erst-/Zweitstimme,Geschlecht,Geburtsjahresgruppe
0,Bund,1,Summe,Summe
1,Bund,1,Summe,1997 – 2003
2,Bund,1,Summe,1987 – 1996
3,Bund,1,Summe,1977 – 1986
4,Bund,1,Summe,1962 – 1976
5,Bund,1,Summe,1952 – 1961
6,Bund,1,Summe,1951 und früher
7,Bund,1,m,Summe
8,Bund,1,m,1997 – 2003
9,Bund,1,m,1987 – 1996


In [15]:
state_statistic_details = select_state_statistic_detail_rows(
    normalized_state_statistics
)
print(f"Verwendete Detailzeilen: {len(state_statistic_details):,}")
display(state_statistic_details.head(20))

Verwendete Detailzeilen: 384


,Land,Erst-/Zweitstimme,Geschlecht,Geburtsjahresgruppe,Summe,Ungültig,CDU,SPD,AfD,FDP,DIE LINKE,GRÜNE,CSU,Sonstige,dar. FREIE WÄHLER,dar. dieBasis,state,voteType,gender,ageGroup
50,SH,1,m,1997 – 2003,66529,694,9478,12901,3319,14435,3857,16164,,5681,863,554,Schleswig-Holstein,1,m,18-24
51,SH,1,m,1987 – 1996,96579,578,14851,20904,10100,15494,5186,20209,,9256,1683,1519,Schleswig-Holstein,1,m,25-34
52,SH,1,m,1977 – 1986,107371,549,23770,27281,13607,10462,4101,17679,,9922,2686,2488,Schleswig-Holstein,1,m,35-44
53,SH,1,m,1962 – 1976,245329,1772,66001,68443,24942,22852,6835,37377,,17108,4746,3984,Schleswig-Holstein,1,m,45-54
54,SH,1,m,1952 – 1961,150261,1634,38538,52157,11096,10622,3944,24632,,7639,2138,1747,Schleswig-Holstein,1,m,55-64
55,SH,1,m,1951 und früher,189819,2868,75673,65744,7620,13152,2630,18538,,3594,781,587,Schleswig-Holstein,1,m,65+
57,SH,1,w,1997 – 2003,65832,349,7852,13255,1893,9861,4674,22686,,5263,1255,864,Schleswig-Holstein,1,w,18-24
58,SH,1,w,1987 – 1996,99872,629,15001,26221,8045,10762,5318,25015,,8881,2287,1897,Schleswig-Holstein,1,w,25-34
59,SH,1,w,1977 – 1986,121165,650,26874,32453,7574,11222,4495,25966,,11933,3261,3107,Schleswig-Holstein,1,w,35-44
60,SH,1,w,1962 – 1976,255806,1541,61120,75089,13327,23409,8519,52590,,20211,4930,6486,Schleswig-Holstein,1,w,45-54


## 9. Statistik in lange Parteizeilen umformen

Die veröffentlichten absoluten Statistikwerte werden zunächst nur umgeformt. Noch werden sie nicht als amtliche Stimmenzahlen verwendet.

In [16]:
state_statistic_votes = reshape_state_statistic_votes(state_statistic_details)
print(f"Statistikzellen: {len(state_statistic_votes):,}")
display(state_statistic_votes.head(30))

Statistikzellen: 3,840


,state,voteType,party,gender,ageGroup,statisticVotes
0,Baden-Württemberg,1,AfD,m,18-24,17085.0
1,Baden-Württemberg,1,AfD,m,25-34,44896.0
2,Baden-Württemberg,1,AfD,m,35-44,63967.0
3,Baden-Württemberg,1,AfD,m,45-54,119694.0
4,Baden-Württemberg,1,AfD,m,55-64,61079.0
5,Baden-Württemberg,1,AfD,m,65+,38239.0
6,Baden-Württemberg,1,AfD,w,18-24,8893.0
7,Baden-Württemberg,1,AfD,w,25-34,29727.0
8,Baden-Württemberg,1,AfD,w,35-44,39304.0
9,Baden-Württemberg,1,AfD,w,45-54,70848.0


## 10. Gerundete Statistikwerte in Anteile umrechnen

Die repräsentative Statistik kann durch Rundung und Methodik vom amtlichen Endergebnis abweichen. Deshalb wird jede Zelle durch die statistische Summe ihrer Partei im jeweiligen Land und für die jeweilige Stimmenart geteilt.

Diese Anteile werden später mit den amtlichen Wahlkreis- und Wahlartsummen multipliziert. Die absoluten Statistikwerte werden also nicht als Endsumme übernommen.

In [17]:
demographic_profiles = calculate_demographic_profiles(state_statistic_votes)

display(demographic_profiles.head(30))

share_checks = (
    demographic_profiles.groupby(["state", "voteType", "party"])["share"]
    .sum()
    .sub(1.0)
    .abs()
)
print(f"Größte Abweichung einer Anteils-Summe von 1: {share_checks.max():.12g}")

,state,voteType,party,gender,ageGroup,statisticVotes,statisticTotal,share
0,Baden-Württemberg,1,AfD,m,18-24,17085.0,561058.0,0.030451
1,Baden-Württemberg,1,AfD,m,25-34,44896.0,561058.0,0.080020
2,Baden-Württemberg,1,AfD,m,35-44,63967.0,561058.0,0.114011
3,Baden-Württemberg,1,AfD,m,45-54,119694.0,561058.0,0.213336
4,Baden-Württemberg,1,AfD,m,55-64,61079.0,561058.0,0.108864
5,Baden-Württemberg,1,AfD,m,65+,38239.0,561058.0,0.068155
6,Baden-Württemberg,1,AfD,w,18-24,8893.0,561058.0,0.015850
7,Baden-Württemberg,1,AfD,w,25-34,29727.0,561058.0,0.052984
8,Baden-Württemberg,1,AfD,w,35-44,39304.0,561058.0,0.070053
9,Baden-Württemberg,1,AfD,w,45-54,70848.0,561058.0,0.126276


Größte Abweichung einer Anteils-Summe von 1: 0


In [18]:
# Beispiel: veröffentlichte statistische Summe und amtliche Summe nebeneinander.
statistic_totals = (
    demographic_profiles[["state", "voteType", "party", "statisticTotal"]]
    .drop_duplicates()
)
official_totals = (
    district_totals.groupby(["state", "voteType", "party"], as_index=False)["votes"]
    .sum()
    .rename(columns={"votes": "officialTotal"})
)
total_comparison = official_totals.merge(
    statistic_totals,
    on=["state", "voteType", "party"],
    how="inner",
)
total_comparison["difference"] = (
    total_comparison["officialTotal"] - total_comparison["statisticTotal"]
)
display(total_comparison.reindex(total_comparison["difference"].abs().sort_values(ascending=False).index).head(30))

,state,voteType,party,officialTotal,statisticTotal,difference
39,Berlin,1,Sonstige,251.0,145636.0,-145385.0
47,Berlin,2,SPD,374413.0,428288.0,-53875.0
38,Berlin,1,SPD,366569.0,417161.0,-50592.0
46,Berlin,2,GRÜNE,370735.0,408531.0,-37796.0
37,Berlin,1,GRÜNE,346880.0,380581.0,-33701.0
44,Berlin,2,FDP,136998.0,165936.0,-28938.0
35,Berlin,1,FDP,106292.0,129678.0,-23386.0
34,Berlin,1,DIE LINKE,238776.0,260243.0,-21467.0
43,Berlin,2,DIE LINKE,194010.0,209053.0,-15043.0
20,Bayern,1,FREIE WÄHLER,587357.0,577845.0,9512.0


## 11. Optionales Bundesmuster für Brief- und Urnenwahl

Diese dritte Datei ist nicht pro Wahlkreis. Sie enthält ein bundesweites Muster nach Partei, Geschlecht, Alter und Brief-/Urnenwahl. IPF benutzt es nur als Ausgangsmuster.

Ist kein Pfad eingetragen, wird ein neutrales Ausgangsmuster verwendet. Die bekannten Landesränder werden in beiden Fällen exakt angepasst.

In [19]:
federal_method_seed = None

if FEDERAL_METHOD_DEMOGRAPHICS_CSV is None:
    print("Keine optionale Bundesdatei angegeben. Es wird ein neutrales Ausgangsmuster verwendet.")
else:
    raw_federal_method_statistics = read_local_csv(
        FEDERAL_METHOD_DEMOGRAPHICS_CSV,
        comment="#",
    )
    display(raw_federal_method_statistics.head(20))

    normalized_federal_method_statistics = normalize_federal_method_rows(
        raw_federal_method_statistics
    )
    display(normalized_federal_method_statistics.head(20))

    federal_method_details = select_federal_method_detail_rows(
        normalized_federal_method_statistics
    )
    federal_method_votes = reshape_federal_method_votes(federal_method_details)
    display(federal_method_votes.head(30))

    federal_method_seed = calculate_federal_method_weights(federal_method_votes)
    display(federal_method_seed.head(30))

Keine optionale Bundesdatei angegeben. Es wird ein neutrales Ausgangsmuster verwendet.


## 12. Landesprofile mit IPF berechnen

Für jede Partei und Stimmenart werden zwei bekannte Ränder zusammengeführt:

- die demografischen Anteile aus der Landesstatistik;
- die amtlichen Brief- und Urnenstimmen aus der Wahlbezirksdatei.

Parteien ohne eigene Statistik verwenden das Profil von `Sonstige`. Nur wenn auch dieses fehlt, wird gleichverteilt.

In [20]:
state_method_profiles = build_state_method_profiles(
    district_totals,
    demographic_profiles,
    federal_method_seed,
)

print(f"Profilzeilen: {len(state_method_profiles):,}")
display(
    state_method_profiles[
        ["party", "demographicProfileSource", "methodSeedSource"]
    ].drop_duplicates().sort_values(["demographicProfileSource", "party"])
)
display(state_method_profiles.head(30))

Profilzeilen: 15,288


,party,demographicProfileSource,methodSeedSource
2400,B*,other-statistics,independent-method-fallback
1176,BP,other-statistics,independent-method-fallback
600,BÜNDNIS21,other-statistics,independent-method-fallback
24,BÜRGERBEWEGUNG,other-statistics,independent-method-fallback
48,BüSo,other-statistics,independent-method-fallback
72,Bündnis C,other-statistics,independent-method-fallback
720,DKP,other-statistics,independent-method-fallback
144,DiB,other-statistics,independent-method-fallback
2496,Die Grauen,other-statistics,independent-method-fallback
168,Die Humanisten,other-statistics,independent-method-fallback


,state,voteType,party,electionMethod,gender,ageGroup,share,stateMethodVotes,fittedStateVotes,demographicProfileSource,methodSeedSource
0,Baden-Württemberg,1,AfD,in-person,m,18-24,0.030451,369063.0,11238.483998,party-statistics,independent-method-fallback
1,Baden-Württemberg,1,AfD,in-person,m,25-34,0.080020,369063.0,29532.512589,party-statistics,independent-method-fallback
2,Baden-Württemberg,1,AfD,in-person,m,35-44,0.114011,369063.0,42077.384016,party-statistics,independent-method-fallback
3,Baden-Württemberg,1,AfD,in-person,m,45-54,0.213336,369063.0,78734.510019,party-statistics,independent-method-fallback
4,Baden-Württemberg,1,AfD,in-person,m,55-64,0.108864,369063.0,40177.662518,party-statistics,independent-method-fallback
5,Baden-Württemberg,1,AfD,in-person,m,65+,0.068155,369063.0,25153.549289,party-statistics,independent-method-fallback
6,Baden-Württemberg,1,AfD,in-person,w,18-24,0.015850,369063.0,5849.800304,party-statistics,independent-method-fallback
7,Baden-Württemberg,1,AfD,in-person,w,25-34,0.052984,369063.0,19554.370138,party-statistics,independent-method-fallback
8,Baden-Württemberg,1,AfD,in-person,w,35-44,0.070053,369063.0,25854.104481,party-statistics,independent-method-fallback
9,Baden-Württemberg,1,AfD,in-person,w,45-54,0.126276,369063.0,46603.694135,party-statistics,independent-method-fallback


## 13. Landesprofile auf die amtlichen Wahlkreissummen anwenden

Nun gilt für jede Wahlkreis-Partei-Wahlart-Kombination:

`geschätzte Stimmen = amtliche Wahlkreissumme × berechneter Profilanteil`

Die Summe aller erzeugten Detailzeilen bleibt dabei gleich der amtlichen Ausgangssumme.

In [21]:
first_district_totals = district_totals[district_totals["voteType"] == "1"].copy()
second_district_totals = district_totals[district_totals["voteType"] == "2"].copy()

first_votes = distribute_district_votes(first_district_totals, state_method_profiles)
second_votes = distribute_district_votes(second_district_totals, state_method_profiles)
all_votes = [*first_votes, *second_votes]

print(f"Erststimmen-Einträge: {len(first_votes):,}")
print(f"Zweitstimmen-Einträge: {len(second_votes):,}")
display(pd.DataFrame([asdict(entry) for entry in first_votes[:30]]))

Erststimmen-Einträge: 79,152
Zweitstimmen-Einträge: 167,796


,districtId,state,gender,ageGroup,party,voteType,electionMethod,votes
0,1,Schleswig-Holstein,m,18-24,AfD,1,in-person,231.776843
1,1,Schleswig-Holstein,m,25-34,AfD,1,in-person,705.316696
2,1,Schleswig-Holstein,m,35-44,AfD,1,in-person,950.222207
3,1,Schleswig-Holstein,m,45-54,AfD,1,in-person,1741.783073
4,1,Schleswig-Holstein,m,55-64,AfD,1,in-person,774.870699
5,1,Schleswig-Holstein,m,65+,AfD,1,in-person,532.130022
6,1,Schleswig-Holstein,w,18-24,AfD,1,in-person,132.194506
7,1,Schleswig-Holstein,w,25-34,AfD,1,in-person,561.809190
8,1,Schleswig-Holstein,w,35-44,AfD,1,in-person,528.917689
9,1,Schleswig-Holstein,w,45-54,AfD,1,in-person,930.668872


## 14. Bekannte Summen prüfen

Die Validierung vergleicht die erzeugten Einträge wieder mit den amtlichen Wahlkreis-/Wahlartsummen und mit den berechneten demografischen Landesrändern.

In [22]:
validation = validate_vote_entries(
    all_votes,
    district_totals,
    state_method_profiles,
)
validation

ValidationReport(entryCount=246948, sourceGroupCount=20579, maxDistrictMethodError=4.547473508864641e-13, maxStateDemographicError=3.703746642713668e-09)

## 15. JSON-Dateien schreiben

Erst nach allen sichtbaren Prüfungen werden die beiden Dateien gespeichert.

In [23]:
first_votes_path = write_vote_entries(
    first_votes,
    OUTPUT_DIRECTORY / "first_votes.json",
)
second_votes_path = write_vote_entries(
    second_votes,
    OUTPUT_DIRECTORY / "second_votes.json",
)

first_votes_path, second_votes_path

(PosixPath('/home/nomikron/WebstormProjects/mach-dir-dein-bundestag/scripts/data/generated/first_votes.json'),
 PosixPath('/home/nomikron/WebstormProjects/mach-dir-dein-bundestag/scripts/data/generated/second_votes.json'))